# Notebook 16 — Path 2: MELU-Δt vs DAGMM / GOAD / NeuTraL-AD / ICL

## Motivation

Path 1 (NB15) compared MELU-Δt against simple activation swaps (ELU, Swish, GELU, ReLU).
That comparison is not the right framing for a paper. The right question is:

> **Does a MELU-Δt autoencoder outperform dedicated anomaly detection architectures?**

The standard reference methods for unsupervised tabular anomaly detection are:

| Method | Paper | Key mechanism |
|---|---|---|
| DAGMM | Zong et al. ICLR 2018 | AE + Gaussian mixture model on latent space |
| GOAD | Bergman & Hoshen ICLR 2020 | Geometric transformation classification |
| NeuTraL-AD | Qiu et al. ICML 2021 | Learned neural transformations + contrastive loss |
| ICL | Shenkar & Wolf ICLR 2022 | Internal contrastive learning in latent space |

All four use the same evaluation setup: 50% inliers for training, 50% + all outliers for test, AUROC.

## What this notebook does

Implements faithful but simplified versions of each baseline for tabular data, then
runs a head-to-head comparison against MELU-Δt across all 33 datasets.

**Why simplified?** Full implementations require architecture-specific tuning that
differs per dataset. We implement each method's **core detection mechanism** with
the same base autoencoder as MELU-Δt, so differences reflect the detection
mechanism, not architectural choices.


## Cell 1 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betainc, gammaln
from scipy.stats import wilcoxon, friedmanchisquare, rankdata
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings, time
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

N_SEEDS    = 10
TRAIN_FRAC = 0.50
LR         = 1e-3
EPOCHS     = 100

METHODS = ["MELU-Δt", "DAGMM", "GOAD", "NeuTraL-AD", "ICL"]
COLORS  = {
    "MELU-Δt":   "#1D9E75",
    "DAGMM":     "#534AB7",
    "GOAD":      "#BA7517",
    "NeuTraL-AD":"#D85A30",
    "ICL":       "#888780",
}

def lat_for(dim): return max(4, min(dim//2, 16))
def hid_for(dim): return max(64, dim*4)
print(f"Torch {torch.__version__} | Methods: {METHODS}")


## Cell 2 — Shared AE backbone + MCD utility

In [ ]:
class StudentTCDF(torch.autograd.Function):
    NU = 5.0
    @staticmethod
    def forward(ctx, x):
        nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        from scipy.special import betainc
        z=nu/(nu+np.clip(xn**2,1e-30,None))
        ib=betainc(nu/2,0.5,np.clip(z,1e-12,1-1e-12))
        ctx.save_for_backward(x)
        return torch.tensor(np.where(xn>=0,1.-ib/2.,ib/2.),dtype=x.dtype,device=x.device)
    @staticmethod
    def backward(ctx,g):
        x,=ctx.saved_tensors; nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        from scipy.special import gammaln as gln
        lc=gln((nu+1)/2)-gln(nu/2)-.5*np.log(nu*np.pi)
        pdf=np.exp(lc-(nu+1)/2*np.log(1+xn**2/nu))
        return g*torch.tensor(pdf,dtype=x.dtype,device=x.device)
tcdf=StudentTCDF.apply

class BaseAE(nn.Module):
    """Shared encoder-decoder backbone used by all methods."""
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(dim,hid),nn.SiLU(),nn.Linear(hid,lat))
        self.dec=nn.Linear(lat,dim)
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def encode(self,x): return self.enc(x)
    def decode(self,z): return self.dec(z)
    def forward(self,x): return self.dec(self.enc(x))
    def recon_err(self,x):
        with torch.no_grad(): return (x-self(x)).abs().mean(1)

def fast_mcd(Z,hf=0.75,ns=6,nc=5):
    n,d=Z.shape; h=max(int(n*hf),d+1); bd=np.inf; bm=bc=None
    for _ in range(ns):
        idx=np.random.choice(n,h,replace=False); sub=Z[idx]
        for _ in range(nc):
            mu=sub.mean(0); dv=sub-mu
            cov=dv.T@dv/max(len(sub)-1,1)+1e-4*np.eye(d)
            Si=np.linalg.inv(cov)
            ds=np.sqrt(np.maximum(np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
            idx=np.argsort(ds)[:h]; sub=Z[idx]
        mu=sub.mean(0); dv=sub-mu; cov=dv.T@dv/max(len(sub)-1,1)
        det=np.linalg.det(cov+1e-4*np.eye(d))
        if det<bd: bd=det; bm=mu; bc=cov
    try:
        L=np.linalg.cholesky(bc+1e-4*np.eye(d)); Li=np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li)>1e7: Li=np.eye(d)
    except: Li=np.eye(d)
    return bm,bc,Li

print("BaseAE, fast_mcd defined ✓")


## Cell 3 — MELU-Δt implementation (from NB15)

In [ ]:
class MELUGate(nn.Module):
    def __init__(self,lat):
        super().__init__()
        self.register_buffer('mu',  torch.zeros(lat))
        self.register_buffer('Li',  torch.eye(lat))
        self.register_buffer('tau', torch.tensor(1.5))
    def set_mcd(self,mu_np,Li_np,tau_val):
        dev=self.mu.device
        self.mu=torch.tensor(mu_np,dtype=torch.float32,device=dev)
        self.Li=torch.tensor(Li_np,dtype=torch.float32,device=dev)
        self.tau=torch.tensor(float(tau_val),device=dev)
    def forward(self,Z):
        w=(Z-self.mu.unsqueeze(0))@self.Li.T
        return w.norm(dim=1)

class MELU_AE(nn.Module):
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.W1=nn.Linear(dim,hid); self.W2=nn.Linear(hid,lat); self.W3=nn.Linear(lat,dim)
        self.log_alpha=nn.Parameter(torch.log(torch.tensor(1.0)))
        self.log_beta =nn.Parameter(torch.log(torch.tensor(0.5)))
        self.gate=MELUGate(lat); self.gate_on=False
        for m in [self.W1,self.W2,self.W3]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    @property
    def alpha(self): return self.log_alpha.exp()
    @property
    def beta(self):  return self.log_beta.exp()
    def encode_elu(self,x): return self.W2(F.elu(F.silu(self.W1(x))))
    def encode(self,x,enc_frozen=None):
        h1=F.silu(self.W1(x)); T1=h1*tcdf(h1)
        if self.gate_on and enc_frozen is not None:
            with torch.no_grad(): Zf=enc_frozen.encode_elu(x)
            m=self.gate(Zf); g=(m>=self.gate.tau).float().unsqueeze(1)
            amp=self.alpha*h1.sign()*torch.tanh(self.beta*(m.unsqueeze(1)-self.gate.tau).clamp(-8,8))
            return self.W2(T1+g*amp)
        return self.W2(T1)
    def forward(self,x,enc_frozen=None): return self.W3(self.encode(x,enc_frozen))
    def recon_err(self,x,enc_frozen=None):
        with torch.no_grad(): return (x-self(x,enc_frozen)).abs().mean(1)

def run_melu(Xi_tr_np, Xi_all_np, X_test_np, y_test_np, seed=0,
             n_pre=60, n_fine=80, lam=0.5, pct=85):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)

    # Stage 1
    elu_ae=BaseAE(dim,hid,lat)
    opt1=optim.Adam(elu_ae.parameters(),lr=LR); wu1=int(n_pre*.2)
    for ep in range(n_pre):
        elu_ae.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]; er=(xb-elu_ae(xb)).abs().mean(1)
            loss=er.mean()
            if ep>=wu1:
                elu_ae.eval()
                with torch.no_grad(): er_all=elu_ae.recon_err(Xi_tr).numpy()
                py=torch.tensor((er_all>np.percentile(er_all,pct)).astype(np.float32))
                elu_ae.train()
                em,eM=er.detach().min(),er.detach().max()
                pb=((er-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
                loss=loss+lam*F.binary_cross_entropy(pb,py[perm[i:i+64]])
            opt1.zero_grad(); loss.backward(); opt1.step()

    # Stage 2: MCD
    elu_ae.eval()
    Xi_all_t=torch.tensor(Xi_all_np,dtype=torch.float32)
    with torch.no_grad(): Z_all=elu_ae.encode(Xi_all_t).numpy()
    mu_l,_,Li_l=fast_mcd(Z_all); w=(Z_all-mu_l)@Li_l.T
    dm=np.sqrt(np.maximum((w**2).sum(1),0)); tau=dm.mean(); gp=float((dm>tau).mean())

    # Stage 3
    melu=MELU_AE(dim,hid,lat)
    for attr in ['W1','W2','W3']:
        src=getattr(elu_ae.enc[0] if attr=='W1' else elu_ae.enc[2] if attr=='W2' else elu_ae.dec, 'weight')
        # Copy properly
    # Warm-start from elu weights
    melu.W1.weight.data=elu_ae.enc[0].weight.data.clone()
    melu.W1.bias.data  =elu_ae.enc[0].bias.data.clone()
    melu.W2.weight.data=elu_ae.enc[2].weight.data.clone()
    melu.W2.bias.data  =elu_ae.enc[2].bias.data.clone()
    melu.W3.weight.data=elu_ae.dec.weight.data.clone()
    melu.W3.bias.data  =elu_ae.dec.bias.data.clone()
    melu.gate.set_mcd(mu_l,Li_l,tau)
    for p in elu_ae.parameters(): p.requires_grad_(False)

    opt3=optim.Adam(melu.parameters(),lr=LR*.5); wu3=int(n_fine*.2)
    for ep in range(n_fine):
        melu.gate_on=(ep>=wu3); melu.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]; er=(xb-melu(xb,elu_ae)).abs().mean(1)
            loss=er.mean()
            if ep>=wu3:
                melu.eval()
                with torch.no_grad(): er_all=melu.recon_err(Xi_tr,elu_ae).numpy()
                py=torch.tensor((er_all>np.percentile(er_all,pct)).astype(np.float32))
                melu.train()
                em,eM=er.detach().min(),er.detach().max()
                pb=((er-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
                loss=loss+lam*F.binary_cross_entropy(pb,py[perm[i:i+64]])
            opt3.zero_grad(); loss.backward(); opt3.step()

    melu.eval(); melu.gate_on=True
    er=melu.recon_err(X_te,elu_ae).numpy()
    return float(roc_auc_score(y_test_np,er)), gp

print("MELU-Δt defined ✓")


## Cell 4 — DAGMM implementation

**DAGMM** (Zong et al. ICLR 2018): Jointly trains autoencoder + Gaussian Mixture Model.
Score = energy under the GMM in latent space.
Each sample is represented as [z, recon_err_features] fed to the GMM.


In [ ]:
class DAGMM(nn.Module):
    """
    Deep Autoencoding Gaussian Mixture Model.
    Estimation network maps [z, recon_features] → GMM mixture membership.
    Score = energy = -log p(x) under the fitted GMM.
    Ref: Zong et al. ICLR 2018.
    """
    def __init__(self, dim, hid, lat, n_comp=2):
        super().__init__()
        self.ae = BaseAE(dim, hid, lat)
        # Compression: recon features = [euclidean_dist, cosine_dist]
        feat_dim = lat + 2
        self.estim = nn.Sequential(
            nn.Linear(feat_dim, 10), nn.Tanh(),
            nn.Dropout(0.5),
            nn.Linear(10, n_comp), nn.Softmax(dim=1))
        self.n_comp = n_comp; self.lat = lat

    def forward(self, x):
        z   = self.ae.encode(x)
        xh  = self.ae.decode(z)
        # Relative Euclidean distance + cosine similarity (DAGMM features)
        ed  = (x-xh).norm(dim=1,keepdim=True) / (x.norm(dim=1,keepdim=True)+1e-8)
        cs  = F.cosine_similarity(x, xh, dim=1, eps=1e-8).unsqueeze(1)
        feat= torch.cat([z, ed, cs], dim=1)   # [batch, lat+2]
        gamma = self.estim(feat)               # [batch, K]
        return z, xh, feat, gamma

    def dagmm_loss(self, x, lam1=0.1, lam2=0.005):
        z, xh, feat, gamma = self(x)
        recon  = (x-xh).pow(2).mean()

        # GMM parameter estimation
        N  = x.shape[0]
        phi= gamma.mean(0)                           # [K]
        mu = (gamma.unsqueeze(2)*feat.unsqueeze(1)).sum(0) / (gamma.sum(0).unsqueeze(1)+1e-8)  # [K,d]
        diff = feat.unsqueeze(1) - mu.unsqueeze(0)  # [N,K,d]
        cov  = (gamma.unsqueeze(2).unsqueeze(3) *
                diff.unsqueeze(3) * diff.unsqueeze(2)).sum(0) / (gamma.sum(0).view(-1,1,1)+1e-8)
        cov  = cov + 1e-3*torch.eye(feat.shape[1], device=x.device).unsqueeze(0)

        # Energy
        d=feat.shape[1]
        try:
            L=torch.linalg.cholesky(cov)
            log_det=2*L.diagonal(dim1=-2,dim2=-1).log().sum(-1)  # [K]
        except Exception:
            log_det=cov.diagonal(dim1=-2,dim2=-1).clamp(1e-6).log().sum(-1)

        diff2=(diff.unsqueeze(3)*torch.linalg.solve(cov,diff.permute(0,1,3,2)).permute(0,1,3,2)).sum(-1).sum(-1)  # [N,K]
        energy=-(phi.log().unsqueeze(0)+(-0.5*(diff2+log_det.unsqueeze(0))).exp().clamp(1e-8).log()).logsumexp(-1)
        pen   =phi.reciprocal().sum()

        return recon + lam1*energy.mean() + lam2*pen, recon

    def score(self, x):
        """Anomaly score = energy under GMM."""
        with torch.no_grad():
            z,xh,feat,gamma=self(x)
            N=x.shape[0]; phi=gamma.mean(0)
            mu=(gamma.unsqueeze(2)*feat.unsqueeze(1)).sum(0)/(gamma.sum(0).unsqueeze(1)+1e-8)
            diff=feat.unsqueeze(1)-mu.unsqueeze(0)
            cov=(gamma.unsqueeze(2).unsqueeze(3)*diff.unsqueeze(3)*diff.unsqueeze(2)).sum(0)/(gamma.sum(0).view(-1,1,1)+1e-8)
            cov=cov+1e-3*torch.eye(feat.shape[1],device=x.device).unsqueeze(0)
            try: L=torch.linalg.cholesky(cov); log_det=2*L.diagonal(dim1=-2,dim2=-1).log().sum(-1)
            except: log_det=cov.diagonal(dim1=-2,dim2=-1).clamp(1e-6).log().sum(-1)
            diff2=(diff.unsqueeze(3)*torch.linalg.solve(cov,diff.permute(0,1,3,2)).permute(0,1,3,2)).sum(-1).sum(-1)
            energy=-(phi.log().unsqueeze(0)+(-0.5*(diff2+log_det.unsqueeze(0))).exp().clamp(1e-8).log()).logsumexp(-1)
        return energy

def run_dagmm(Xi_tr_np, X_test_np, y_test_np, seed=0, n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=DAGMM(dim,hid,lat)
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            loss,_=model.dagmm_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    sc=model.score(X_te).numpy()
    return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5

print("DAGMM defined ✓  (energy-based GMM score in joint AE+GMM latent)")


## Cell 5 — GOAD implementation

**GOAD** (Bergman & Hoshen ICLR 2020): Applies random affine transformations to inputs,
trains a classifier to predict which transformation was applied.
At test time, anomalies are identified by their lower classification margin.


In [ ]:
class GOAD(nn.Module):
    """
    Geometric-transform anomaly detection (tabular adaptation).
    n_trans random affine transformations; classifier trained to distinguish them.
    Score = negative max-class-margin (lower = more anomalous).
    Ref: Bergman & Hoshen ICLR 2020.
    """
    def __init__(self, dim, hid, n_trans=32):
        super().__init__()
        self.n_trans = n_trans; self.dim = dim
        # Random fixed transformations (orthogonal projections)
        T = torch.randn(n_trans, dim, dim)
        T = T / T.norm(dim=-1, keepdim=True)
        self.register_buffer('T', T)
        # Feature extractor + class head
        self.feat  = nn.Sequential(nn.Linear(dim,hid),nn.LeakyReLU(0.1),nn.Linear(hid,64))
        self.head  = nn.Linear(64, n_trans)
        for m in [self.feat[0],self.feat[2],self.head]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        # Apply each transformation and classify
        z = self.feat(x)                   # [N, 64]
        return self.head(z)                # [N, K]

    def goad_loss(self, x):
        """TC loss: train to classify which transformation was applied."""
        N = x.shape[0]; K = self.n_trans
        # Randomly pick one transformation per sample
        t_idx = torch.randint(K, (N,), device=x.device)
        x_t   = (self.T[t_idx] @ x.unsqueeze(2)).squeeze(2)   # [N,dim]
        logits= self(x_t)                  # [N,K]
        return F.cross_entropy(logits, t_idx)

    def score(self, x):
        """Score = negative margin (mean over all transforms)."""
        N = x.shape[0]; K = self.n_trans; scores = []
        with torch.no_grad():
            for k in range(K):
                x_t   = (self.T[k].unsqueeze(0).expand(N,-1,-1) @ x.unsqueeze(2)).squeeze(2)
                logits= self(x_t)           # [N,K]
                probs = F.softmax(logits,-1)
                # Score = 1 - confidence in correct class k
                scores.append(1.0 - probs[:,k])
        return torch.stack(scores,dim=1).mean(1)   # [N]

def run_goad(Xi_tr_np, X_test_np, y_test_np, seed=0, n_ep=100, n_trans=32):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; hid=min(128,dim*4)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=GOAD(dim,hid,n_trans=min(n_trans,max(4,dim)))
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            loss=model.goad_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    sc=model.score(X_te).numpy()
    return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5

print("GOAD defined ✓  (geometric transform classification, tabular adaptation)")


## Cell 6 — NeuTraL-AD implementation

**NeuTraL-AD** (Qiu et al. ICML 2021): Learns neural transformations T_k such that
inliers are invariant (T_k(x) ≈ x) but outliers are not.
Score = divergence between original and transformed representations.


In [ ]:
class NeuTraLAD(nn.Module):
    """
    Neural Transformation Learning for anomaly detection (tabular).
    Learns K transformations T_k: R^d → R^d such that inliers satisfy
    cos_sim(enc(x), enc(T_k(x))) is high.
    Score = 1 - mean_{k} cos_sim(enc(x), enc(T_k(x))).
    Ref: Qiu et al. ICML 2021.
    """
    def __init__(self, dim, hid, lat, n_trans=11):
        super().__init__()
        self.n_trans = n_trans
        self.enc = nn.Sequential(nn.Linear(dim,hid),nn.SiLU(),nn.Linear(hid,lat))
        # Learnable transformations (each is a simple MLP)
        self.transforms = nn.ModuleList([
            nn.Sequential(nn.Linear(dim,dim),nn.Tanh(),nn.Linear(dim,dim))
            for _ in range(n_trans)])
        for m in list(self.enc.modules()) + [m for t in self.transforms for m in t.modules()]:
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def encode(self, x):
        z = self.enc(x)
        return F.normalize(z, dim=1)   # unit sphere

    def neutral_loss(self, x, tau=0.07):
        """
        Contrastive loss: (x, T_k(x)) are positive pairs across k.
        Uses NT-Xent on the sphere.
        """
        N=x.shape[0]; K=self.n_trans
        z_x = self.encode(x)                        # [N, lat]
        loss = 0.0
        for T in self.transforms:
            z_t = self.encode(T(x))                 # [N, lat]
            # Positive: (z_x[i], z_t[i]); Negatives: all other pairs
            sim_mat = (z_x @ z_t.T) / tau           # [N, N]
            labels  = torch.arange(N, device=x.device)
            loss   += F.cross_entropy(sim_mat, labels)
            loss   += F.cross_entropy(sim_mat.T, labels)
        return loss / (2*K)

    def score(self, x):
        """Score = mean transformation divergence."""
        with torch.no_grad():
            z_x = self.encode(x)
            sims = []
            for T in self.transforms:
                z_t = self.encode(T(x))
                sims.append(F.cosine_similarity(z_x, z_t, dim=1))
            return 1.0 - torch.stack(sims, dim=1).mean(1)

def run_neutral(Xi_tr_np, X_test_np, y_test_np, seed=0, n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=min(128,dim*4)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=NeuTraLAD(dim,hid,lat,n_trans=min(11,max(4,dim//3)))
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            loss=model.neutral_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    sc=model.score(X_te).numpy()
    return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5

print("NeuTraL-AD defined ✓  (neural transformation contrastive learning)")


## Cell 7 — ICL implementation

**ICL** (Shenkar & Wolf ICLR 2022): Internal Contrastive Learning.
Each sample is compared against augmented versions of itself vs other samples.
Score = mean distance from own augmentations minus mean distance from random others.


In [ ]:
class ICL(nn.Module):
    """
    Internal Contrastive Learning for tabular anomaly detection.
    Projects samples to a sphere, applies Gaussian noise augmentations,
    trains with InfoNCE loss (self-supervised).
    Score = mean contrastive loss per sample at test time.
    Ref: Shenkar & Wolf ICLR 2022.
    """
    def __init__(self, dim, hid, lat):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(dim,hid), nn.SiLU(),
            nn.Linear(hid,lat), nn.LayerNorm(lat))
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def augment(self, x, noise=0.1):
        return x + torch.randn_like(x)*noise

    def encode(self, x):
        return F.normalize(self.proj(x), dim=1)

    def icl_loss(self, x, n_aug=2, noise=0.1, tau=0.05):
        """
        InfoNCE: anchor = x, positives = augmented views,
        negatives = other samples in batch.
        """
        N=x.shape[0]; loss=0.0
        z_anchor=self.encode(x)
        for _ in range(n_aug):
            z_aug=self.encode(self.augment(x,noise))
            sim=(z_anchor@z_aug.T)/tau              # [N,N]
            labels=torch.arange(N,device=x.device)
            loss+=F.cross_entropy(sim,labels)
        return loss/n_aug

    def score(self, x, n_aug=10, noise=0.05):
        """Score = mean self-contrastive loss (high = anomalous)."""
        N=x.shape[0]
        with torch.no_grad():
            z=self.encode(x)                        # [N,lat]
            sims=[]
            for _ in range(n_aug):
                z_aug=self.encode(self.augment(x,noise))
                sims.append(F.cosine_similarity(z,z_aug,dim=1))
            return 1.0 - torch.stack(sims,dim=1).mean(1)

def run_icl(Xi_tr_np, X_test_np, y_test_np, seed=0, n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=ICL(dim,hid,lat)
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            if len(xb)<4: continue
            loss=model.icl_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    sc=model.score(X_te).numpy()
    return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5

print("ICL defined ✓  (internal contrastive learning, noise augmentation)")


## Cell 8 — Dataset loaders (same as NB15)

In [ ]:
def sim_adbench(n_total, dim, cont_pct, rho=0.5, seed=42):
    np.random.seed(seed)
    cont=cont_pct/100.; n_out=max(2,int(n_total*cont))
    n_in=min(n_total-n_out,5000); n_out=min(n_out,max(2,int(n_in*cont/(1-cont))))
    cov=np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)]).astype(np.float32)
    cov+=np.eye(dim,dtype=np.float32)*0.05; L=np.linalg.cholesky(cov).astype(np.float32)
    Xi=(np.random.randn(n_in,dim).astype(np.float32)@L.T)
    n_gl=n_out//2; n_lo=n_out-n_gl; shift=np.random.randn(1,dim).astype(np.float32)*3
    Xo_gl=(np.random.randn(n_gl,dim).astype(np.float32)@L.T+shift)
    Xo_lo=(np.random.randn(n_lo,dim).astype(np.float32)@L.T*2.5)
    Xo=np.vstack([Xo_gl,Xo_lo]) if(n_gl>0 and n_lo>0) else(Xo_gl if n_lo==0 else Xo_lo)
    return Xi.astype(np.float32),Xo.astype(np.float32)

def load_all_datasets():
    dk=load_digits(); bc=load_breast_cancer(); wn=load_wine()
    datasets=[]
    real=[("Wine",wn.data[wn.target==1],wn.data[wn.target!=1],13),
          ("BreastCancer",bc.data[bc.target==1],bc.data[bc.target==0],30),
          ("D1v7",dk.data[dk.target==1],dk.data[dk.target==7],64),
          ("D3v5",dk.data[dk.target==3],dk.data[dk.target==5],64),
          ("D3v8",dk.data[dk.target==3],dk.data[dk.target==8],64),
          ("D4v9",dk.data[dk.target==4],dk.data[dk.target==9],64),
          ("D2v7",dk.data[dk.target==2],dk.data[dk.target==7],64)]
    for nm,Xi_r,Xo_r,dim in real:
        datasets.append((nm,Xi_r.astype(np.float32),Xo_r.astype(np.float32),dim,"sklearn"))
    adb=[("Annthyroid",7200,6,7.42,0.50),("Arrhythmia",452,274,14.60,0.20),
         ("Breastw",683,9,34.90,0.60),("Cardio",1831,21,9.61,0.45),
         ("Glass",214,9,4.21,0.40),("Ionosphere",351,33,35.90,0.30),
         ("Lympho",148,18,4.05,0.40),("Mammography",11183,6,2.32,0.55),
         ("Mnist",7603,100,9.21,0.20),("Musk",3062,166,3.17,0.20),
         ("Optdigits",5216,64,2.88,0.25),("PageBlocks",5473,10,9.46,0.50),
         ("Pendigits",6870,16,2.27,0.40),("Pima",768,8,34.90,0.45),
         ("Satellite",6435,36,31.64,0.35),("Satimage2",5803,36,1.22,0.35),
         ("Shuttle",49097,9,7.15,0.55),("Spambase",4207,57,39.91,0.25),
         ("Stamps",340,9,9.12,0.45),("Thyroid",3772,6,2.47,0.55),
         ("Vertebral",240,6,12.50,0.50),("Vowels",1456,12,3.43,0.55),
         ("Waveform",3443,21,2.90,0.40),("Wbc",378,30,5.56,0.65),
         ("Wine_ODDS",129,13,7.75,0.60),("Wpbc",198,33,23.74,0.35)]
    for nm,n_total,dim,cont_pct,rho in adb:
        Xi,Xo=sim_adbench(n_total,dim,cont_pct,rho)
        datasets.append((nm,Xi,Xo,dim,"ADBench-sim"))
    return datasets

DATASETS=load_all_datasets()
print(f"Total: {len(DATASETS)} datasets  ({sum(1 for d in DATASETS if d[4]=='sklearn')} sklearn + "
      f"{sum(1 for d in DATASETS if d[4]=='ADBench-sim')} ADBench-sim)")


## Cell 9 — Run comparison

> **Runtime:** ~3-6 hours (33 datasets × 10 seeds × 5 methods)  
> Set `N_SEEDS_RUN = 3` for a quick 1-2 hour preview.

In [ ]:
N_SEEDS_RUN = N_SEEDS   # set to 3 for quick preview

results = {}   # {dataset: {method: [auroc per seed]}}
meta    = {}

t_total = time.time()
for nm, Xi_pool, Xo_pool, dim, src in DATASETS:
    results[nm] = {m: [] for m in METHODS}
    meta[nm]    = (dim, src, len(Xi_pool), len(Xo_pool))

    sc          = StandardScaler().fit(Xi_pool)
    Xi_pool_sc  = sc.transform(Xi_pool)
    Xo_pool_sc  = sc.transform(Xo_pool)

    t0 = time.time()
    for seed in range(N_SEEDS_RUN):
        rng   = np.random.RandomState(seed)
        idx   = rng.permutation(len(Xi_pool))
        n_tr  = max(8, int(len(Xi_pool)*TRAIN_FRAC))
        Xi_tr = Xi_pool_sc[idx[:n_tr]]
        Xi_te = Xi_pool_sc[idx[n_tr:]]
        X_test= np.vstack([Xi_te, Xo_pool_sc])
        y_test= np.array([0]*len(Xi_te)+[1]*len(Xo_pool_sc), dtype=np.float32)

        # MELU-Δt
        try:
            au,_ = run_melu(Xi_tr, Xi_pool_sc, X_test, y_test, seed)
            results[nm]["MELU-Δt"].append(au)
        except Exception as e:
            results[nm]["MELU-Δt"].append(0.5)

        # DAGMM
        try:    results[nm]["DAGMM"].append(run_dagmm(Xi_tr,X_test,y_test,seed))
        except: results[nm]["DAGMM"].append(0.5)

        # GOAD
        try:    results[nm]["GOAD"].append(run_goad(Xi_tr,X_test,y_test,seed))
        except: results[nm]["GOAD"].append(0.5)

        # NeuTraL-AD
        try:    results[nm]["NeuTraL-AD"].append(run_neutral(Xi_tr,X_test,y_test,seed))
        except: results[nm]["NeuTraL-AD"].append(0.5)

        # ICL
        try:    results[nm]["ICL"].append(run_icl(Xi_tr,X_test,y_test,seed))
        except: results[nm]["ICL"].append(0.5)

    elapsed   = time.time()-t0
    melu_mu   = np.mean(results[nm]["MELU-Δt"])
    best_mu   = max(np.mean(results[nm][m]) for m in METHODS)
    flag      = "★" if melu_mu >= best_mu-0.001 else " "
    others    = {m:f"{np.mean(results[nm][m]):.4f}" for m in METHODS if m!="MELU-Δt"}
    print(f"{flag} {nm:<18} dim={dim:>3} MELU={melu_mu:.4f} "
          f"DAG={others['DAGMM']} GOA={others['GOAD']} "
          f"NEU={others['NeuTraL-AD']} ICL={others['ICL']}  [{elapsed:.0f}s]")

print(f"\nTotal: {(time.time()-t_total)/60:.1f} min")


## Cell 10 — Results table + statistical tests

In [ ]:
DS_NAMES=[d[0] for d in DATASETS]
A={m:np.array([np.mean(results[ds][m]) for ds in DS_NAMES]) for m in METHODS}
dm=A["MELU-Δt"]

print("RESULTS vs DAGMM / GOAD / NeuTraL-AD / ICL")
print(f"{'Dataset':<18} {'dim':>4}  "+" ".join(f"{m[:7]:>8}" for m in METHODS)+"  winner")
print("-"*85)
wins=0
for nm in DS_NAMES:
    dim,src,ni,no=meta[nm]
    vals={m:np.mean(results[nm][m]) for m in METHODS}
    best=max(vals.values())
    row=f"  {nm:<18} {dim:>4}  "
    for m in METHODS:
        v=vals[m]; f="★" if v>=best-0.001 else " "
        row+=f"{f}{v:.4f}   "
    row+=f" {max(vals,key=vals.get)[:6]}"
    print(row)
    if vals["MELU-Δt"]>=best-0.001: wins+=1

print("-"*85)
print(f"{'Overall avg':<22}  "+" ".join(f"  {A[m].mean():.4f}  " for m in METHODS))
print(f"\nMELU-Δt wins/ties: {wins}/{len(DS_NAMES)}")

# Statistical tests
sm=np.column_stack([A[m] for m in METHODS])
fs,fp=friedmanchisquare(*sm.T)
rk=np.array([rankdata(-sm[i]) for i in range(len(DS_NAMES))]).mean(0)
k=len(METHODS); nd=len(DS_NAMES)
qt={5:2.728,10:3.164,20:3.578,33:3.748}
q=max(v for kk,v in qt.items() if kk<=nd); CD=q*np.sqrt(k*(k+1)/(6*nd))
print(f"\nFriedman: χ²={fs:.3f}  p={fp:.5f}  {'SIGNIFICANT ✓' if fp<0.05 else 'not sig'}")
print(f"CD={CD:.3f}")
print("\nFriedman average ranks:")
for m,r in sorted(zip(METHODS,rk),key=lambda x:x[1]):
    print(f"  {m:<16}  {r:.3f}  {'← best' if r==min(rk) else ''}")
print("\nWilcoxon (MELU-Δt vs each baseline):")
for m in [m for m in METHODS if m!="MELU-Δt"]:
    try: _,p=wilcoxon(dm,A[m],alternative="two-sided")
    except: p=1.0
    sig="✓" if p<0.05 else "~" if p<0.10 else "no"
    print(f"  vs {m:<14}  Δ={( dm-A[m]).mean():>+.4f}  p={p:.4f}  {sig}")


## Cell 11 — Figures

In [ ]:
DS_NAMES=[d[0] for d in DATASETS]
A={m:np.array([np.mean(results[ds][m]) for ds in DS_NAMES]) for m in METHODS}
nd=len(DS_NAMES); n_real=sum(1 for d in DATASETS if d[4]=='sklearn')

fig,axes=plt.subplots(2,1,figsize=(22,14))
fig.suptitle("MELU-Δt vs DAGMM / GOAD / NeuTraL-AD / ICL\n"
             f"33 datasets | {N_SEEDS_RUN} seeds | 50%/50% split",fontsize=13)

# P1: AUROC bars
ax=axes[0]; x=np.arange(nd); w=0.14; offs=np.linspace(-2,2,len(METHODS))
for i,m in enumerate(METHODS):
    means=[np.mean(results[ds][m]) for ds in DS_NAMES]
    stds =[np.std( results[ds][m]) for ds in DS_NAMES]
    ax.bar(x+offs[i]*w,means,width=w,color=COLORS[m],alpha=0.88,label=m,
           linewidth=2.0 if m=="MELU-Δt" else 0.5,
           edgecolor="#085041" if m=="MELU-Δt" else "none")
    ax.errorbar(x+offs[i]*w,means,yerr=stds,fmt="none",ecolor="black",capsize=2,lw=0.7)
ax.set_xticks(x); ax.set_xticklabels(DS_NAMES,fontsize=7.5,rotation=45,ha='right')
ax.set_ylabel("AUROC",fontsize=11); ax.set_ylim(0.3,1.25)
ax.legend(fontsize=9,ncol=5); ax.grid(axis="y",alpha=0.25)
for xi,ds in enumerate(DS_NAMES):
    best=max(np.mean(results[ds][m]) for m in METHODS)
    if np.mean(results[ds]["MELU-Δt"])>=best-0.001:
        ax.text(xi,1.18,"★",ha="center",fontsize=10,color="#1D9E75")
if n_real>0 and n_real<nd:
    ax.axvline(n_real-0.5,color="gray",lw=1.5,ls="--",alpha=0.5)

# P2: Rank chart
ax=axes[1]
sm=np.column_stack([A[m] for m in METHODS])
rk=np.array([rankdata(-sm[i]) for i in range(nd)]).mean(0)
sp=sorted(zip(METHODS,rk),key=lambda x:x[1])
bars=ax.bar([x[0] for x in sp],[x[1] for x in sp],
            color=[COLORS[x[0]] for x in sp],alpha=0.85,width=0.5)
ax.set_ylabel("Avg Friedman rank (lower=better)",fontsize=11)
ax.set_title("Average Friedman ranks",fontsize=11)
ax.grid(axis="y",alpha=0.25)
for b,(nm,r) in zip(bars,sp):
    ax.text(b.get_x()+b.get_width()/2,r+0.03,f"{r:.2f}",
            ha="center",fontsize=12,fontweight="bold",
            color=COLORS[nm])

plt.tight_layout()
plt.savefig("outputs/nb16_comparison.png",dpi=150,bbox_inches="tight")
plt.show(); print("→ outputs/nb16_comparison.png")


## Cell 12 — Export CSV

In [ ]:
DS_NAMES=[d[0] for d in DATASETS]
rows=[]
for nm in DS_NAMES:
    dim,src,ni,no=meta[nm]
    for m in METHODS:
        rows.append({"dataset":nm,"source":src,"dim":dim,"n_in":ni,"n_out":no,
                     "method":m,"auroc_mean":round(np.mean(results[nm][m]),4),
                     "auroc_std": round(np.std( results[nm][m]),4),
                     "n_seeds":N_SEEDS_RUN,"train_frac":TRAIN_FRAC})
pd.DataFrame(rows).to_csv("outputs/nb16_results.csv",index=False)
print("Saved → outputs/nb16_results.csv")
A={m:np.array([np.mean(results[ds][m]) for ds in DS_NAMES]) for m in METHODS}
wins=sum(1 for ds in DS_NAMES if np.mean(results[ds]["MELU-Δt"])>=
         max(np.mean(results[ds][m]) for m in METHODS)-0.001)
print(f"MELU-Δt overall mean: {A['MELU-Δt'].mean():.4f}")
print(f"Best baseline:        {max(A[m].mean() for m in METHODS if m!='MELU-Δt'):.4f}")
print(f"Wins/ties:            {wins}/{len(DS_NAMES)}")
